# Silver Layer: Cleaned Data & SCD Type 2 Tracking

# Overview
This notebook processes raw JSON entries from the Bronze tables, parses nested attributes into clean columns, and applies **Slowly Changing Dimension (SCD) Type 2** logic.

# SCD Type 2 Logic
**effective_start_date**: Timestamp when the record version was ingested.
**effective_end_date**: Timestamp when a record was updated .
**is_current**: Boolean flag indicating active versions.
Uses **Delta Lake `MERGE`** operations to update existing records and append updated histories.

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import get_json_object, col, explode, current_timestamp, lit, expr

# Clean up older silver tables to avoid schema mismatches from previous test runs
old_silver_tables = ["silver_patient", "silver_encounter", "silver_observation", "silver_condition"]
for st in old_silver_tables:
    spark.sql(f"DROP TABLE IF EXISTS {st}")

# --- 1. PATIENT SILVER LAYER (SCD TYPE 2) ---
def process_patient_silver():
    print("--- Building Silver Patient Table ---")
    bronze_patient = spark.read.table("bronze_patient")
    
    # Extract entries array
    df_entries = bronze_patient.select(
        explode(expr("from_json(get_json_object(raw_payload, '$.entry'), 'ARRAY<STRING>')")).alias("entry_json"),
        col("extraction_timestamp")
    )
    
    # Extract domain fields
    df_extracted = df_entries.select(
        get_json_object("entry_json", "$.resource.id").alias("patient_id"),
        get_json_object("entry_json", "$.resource.gender").alias("gender"),
        get_json_object("entry_json", "$.resource.birthDate").alias("birth_date"),
        col("extraction_timestamp").alias("effective_start_date")
    ).filter(col("patient_id").isNotNull()).dropDuplicates(["patient_id"])

    target_table = "silver_patient"
    
    if not spark.catalog.tableExists(target_table):
        df_initial = df_extracted \
            .withColumn("effective_end_date", lit(None).cast("timestamp")) \
            .withColumn("is_current", lit(True))
        df_initial.write.format("delta").mode("overwrite").saveAsTable(target_table)
        print(f"Created table: {target_table}")
    else:
        silver_table = DeltaTable.forName(spark, target_table)
        silver_table.alias("target").merge(
            source=df_extracted.alias("updates"),
            condition="target.patient_id = updates.patient_id AND target.is_current = true"
        ).whenMatchedUpdate(set={
            "is_current": "false",
            "effective_end_date": "updates.effective_start_date"
        }).execute()
        
        df_new = df_extracted.withColumn("effective_end_date", lit(None).cast("timestamp")).withColumn("is_current", lit(True))
        df_new.write.format("delta").mode("append").saveAsTable(target_table)
        print(f"Merged SCD2 into table: {target_table}")

# --- 2. ENCOUNTER SILVER LAYER ---
def process_encounter_silver():
    print("--- Building Silver Encounter Table ---")
    bronze_df = spark.read.table("bronze_encounter")
    
    df_entries = bronze_df.select(
        explode(expr("from_json(get_json_object(raw_payload, '$.entry'), 'ARRAY<STRING>')")).alias("entry_json"),
        col("extraction_timestamp")
    )
    
    df_extracted = df_entries.select(
        get_json_object("entry_json", "$.resource.id").alias("encounter_id"),
        get_json_object("entry_json", "$.resource.subject.reference").alias("patient_ref"),
        get_json_object("entry_json", "$.resource.status").alias("status"),
        get_json_object("entry_json", "$.resource.class.code").alias("class_code"),
        col("extraction_timestamp").alias("effective_start_date")
    ).filter(col("encounter_id").isNotNull()).dropDuplicates(["encounter_id"])

    target_table = "silver_encounter"
    
    if not spark.catalog.tableExists(target_table):
        df_extracted.withColumn("effective_end_date", lit(None).cast("timestamp")) \
                    .withColumn("is_current", lit(True)) \
                    .write.format("delta").mode("overwrite").saveAsTable(target_table)
        print(f"Created table: {target_table}")
    else:
        silver_table = DeltaTable.forName(spark, target_table)
        silver_table.alias("target").merge(
            source=df_extracted.alias("updates"),
            condition="target.encounter_id = updates.encounter_id AND target.is_current = true"
        ).whenMatchedUpdate(set={
            "is_current": "false",
            "effective_end_date": "updates.effective_start_date"
        }).execute()
        
        df_extracted.withColumn("effective_end_date", lit(None).cast("timestamp")) \
                    .withColumn("is_current", lit(True)) \
                    .write.format("delta").mode("append").saveAsTable(target_table)
        print(f"Merged SCD2 into table: {target_table}")

# --- 3. OBSERVATION SILVER LAYER ---
def process_observation_silver():
    print("--- Building Silver Observation Table ---")
    bronze_df = spark.read.table("bronze_observation")
    
    df_entries = bronze_df.select(
        explode(expr("from_json(get_json_object(raw_payload, '$.entry'), 'ARRAY<STRING>')")).alias("entry_json"),
        col("extraction_timestamp")
    )
    
    df_extracted = df_entries.select(
        get_json_object("entry_json", "$.resource.id").alias("observation_id"),
        get_json_object("entry_json", "$.resource.subject.reference").alias("patient_ref"),
        get_json_object("entry_json", "$.resource.code.text").alias("observation_name"),
        get_json_object("entry_json", "$.resource.status").alias("status"),
        col("extraction_timestamp").alias("effective_start_date")
    ).filter(col("observation_id").isNotNull()).dropDuplicates(["observation_id"])

    target_table = "silver_observation"
    
    if not spark.catalog.tableExists(target_table):
        df_extracted.withColumn("effective_end_date", lit(None).cast("timestamp")) \
                    .withColumn("is_current", lit(True)) \
                    .write.format("delta").mode("overwrite").saveAsTable(target_table)
        print(f"Created table: {target_table}")
    else:
        silver_table = DeltaTable.forName(spark, target_table)
        silver_table.alias("target").merge(
            source=df_extracted.alias("updates"),
            condition="target.observation_id = updates.observation_id AND target.is_current = true"
        ).whenMatchedUpdate(set={
            "is_current": "false",
            "effective_end_date": "updates.effective_start_date"
        }).execute()
        
        df_extracted.withColumn("effective_end_date", lit(None).cast("timestamp")) \
                    .withColumn("is_current", lit(True)) \
                    .write.format("delta").mode("append").saveAsTable(target_table)
        print(f"Merged SCD2 into table: {target_table}")

# --- 4. CONDITION SILVER LAYER ---
def process_condition_silver():
    print("--- Building Silver Condition Table ---")
    bronze_df = spark.read.table("bronze_condition")
    
    df_entries = bronze_df.select(
        explode(expr("from_json(get_json_object(raw_payload, '$.entry'), 'ARRAY<STRING>')")).alias("entry_json"),
        col("extraction_timestamp")
    )
    
    df_extracted = df_entries.select(
        get_json_object("entry_json", "$.resource.id").alias("condition_id"),
        get_json_object("entry_json", "$.resource.subject.reference").alias("patient_ref"),
        get_json_object("entry_json", "$.resource.code.text").alias("condition_name"),
        get_json_object("entry_json", "$.resource.clinicalStatus.coding[0].code").alias("clinical_status"),
        col("extraction_timestamp").alias("effective_start_date")
    ).filter(col("condition_id").isNotNull()).dropDuplicates(["condition_id"])

    target_table = "silver_condition"
    
    if not spark.catalog.tableExists(target_table):
        df_extracted.withColumn("effective_end_date", lit(None).cast("timestamp")) \
                    .withColumn("is_current", lit(True)) \
                    .write.format("delta").mode("overwrite").saveAsTable(target_table)
        print(f"Created table: {target_table}")
    else:
        silver_table = DeltaTable.forName(spark, target_table)
        silver_table.alias("target").merge(
            source=df_extracted.alias("updates"),
            condition="target.condition_id = updates.condition_id AND target.is_current = true"
        ).whenMatchedUpdate(set={
            "is_current": "false",
            "effective_end_date": "updates.effective_start_date"
        }).execute()
        
        df_extracted.withColumn("effective_end_date", lit(None).cast("timestamp")) \
                    .withColumn("is_current", lit(True)) \
                    .write.format("delta").mode("append").saveAsTable(target_table)
        print(f"Merged SCD2 into table: {target_table}")

# Run process in order: Patient -> Encounter -> Observation -> Condition
process_patient_silver()
process_encounter_silver()
process_observation_silver()
process_condition_silver()

--- Building Silver Patient Table ---
Created table: silver_patient
--- Building Silver Encounter Table ---
Created table: silver_encounter
--- Building Silver Observation Table ---
Created table: silver_observation
--- Building Silver Condition Table ---
Created table: silver_condition
